In [1]:
#| default_exp 29_beir-concept-and-summary-metadata

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#| export
import os, json, pandas as pd, ast, scipy.sparse as sp, json_repair, numpy as np, re, joblib

from tqdm.auto import tqdm
from typing import Optional, Dict, List, Tuple

from xcai.misc import BEIR_DATASETS
from sugar.core import *

## `Helper functions`

### Load raw data

In [4]:
#| export
def save_raw(fname:str, ids:List, txt:List):
    df = pd.DataFrame({"identifier": ids, "text": txt})
    df.to_csv(fname, index=False)

def load_raw(fname:str):
    df = pd.read_csv(fname, keep_default_na=False, na_filter=False)
    return df["identifier"].tolist(), df["text"].tolist()
    

### Combine dataframe

In [4]:
#| export
def combine_df(dirname:str):
    files = [o for o in os.listdir(dirname) if re.match(r"UHRS_Task_docs-\d+.tsv", o)]
    files = sorted(files, key=lambda x: int(x.split("-")[1][:-4]))
    return pd.concat([pd.read_table(f"{dirname}/{f}") for f in tqdm(files)], axis=0)
    

### Extract concepts

In [5]:
#| export
def convert_string_to_object(df:pd.DataFrame):
    outputs, n_outputs = {}, 0
    for idx, doc, out in tqdm(zip(df["ids"], df["document"], df["raw_model_response"]), total=df.shape[0]):
        if not pd.isna(doc) and not pd.isna(out):
            match = re.match(r"```json(.*)```", out)
            if match is not None:
                out = match[1]
                try:
                    out = json.loads(out)
                except:
                    out = json_repair.loads(out)
                
                outputs[idx] = {
                    "document": doc,
                    "concepts": out[0]["concepts"] if isinstance(out, list) else out["concepts"],
                }
                n_outputs += 1

    print(f"# of empty generations: {df.shape[0] - n_outputs}.")
    return outputs
    

#### Example

In [80]:
data_dir = "/Users/suchith720/Downloads/outputs/"

In [81]:
df = combine_df(data_dir)

  0%|          | 0/4 [00:00<?, ?it/s]

In [106]:
outputs = convert_string_to_object(df)

  0%|          | 0/400000 [00:00<?, ?it/s]

# of empty generations: 28611.


In [101]:
list(outputs)[:10]

['1911_in_Australia',
 '1994_NatWest_Trophy',
 '1844_in_Wales',
 '1934–35_Newport_County_A.F.C._season',
 '1974_Wisconsin_Badgers_football_team',
 '1973_Fresno_State_Bulldogs_football_team',
 "1992_European_Athletics_Indoor_Championships_–_Women's_400_metres",
 '1944_Washington_Huskies_football_team',
 '1964–65_Brentford_F.C._season',
 '140_(number)']

In [104]:
outputs["1964–65_Brentford_F.C._season"]["output"]["concepts"]

[{'concept': '1964 -- 65 english football season',
  'language': 'en',
  'english_translation': None,
  'type': 'event',
  'summary': 'The English football season spanning from 1964 to 1965, covering all competitions in England during that period.',
  'web_repeatability': 'high',
  'variations': ['1964-65 english football season',
   '1964/65 english football season',
   '1964 to 1965 english football season',
   '1964–65 english football season',
   '1964-1965 english football season']},
 {'concept': 'brentford',
  'language': 'en',
  'english_translation': None,
  'type': 'organization',
  'summary': 'Brentford Football Club, a professional football team based in Brentford, London.',
  'web_repeatability': 'high',
  'variations': ['brentford fc',
   'brentford football club',
   'brentford f.c.',
   'brentford club']},
 {'concept': 'football league third division',
  'language': 'en',
  'english_translation': None,
  'type': 'organization',
  'summary': 'The third tier of the English

### Extract matrix

In [6]:
#| export
def get_metadata_matrix_from_label_dict(outputs:Dict, lbl_ids:List, seed:Optional[int]=100):
    np.random.seed(seed)

    str_info = {
        "variations": dict(), "type": dict(), "summary": dict(),
    }
    lbl_mat = {
        "vocab": dict(), "data": [], "indices": [], "indptr": [0],
    }

    for l in tqdm(lbl_ids):
        o = outputs.get(str(l), {})
        concepts = o.get("concepts", [])
        
        for concept in concepts:
            idx = lbl_mat["vocab"].setdefault(concept["concept"], len(lbl_mat["vocab"]))
            lbl_mat["indices"].append(idx)
            lbl_mat["data"].append(1.0)

            str_info["variations"].setdefault(idx, []).extend(concept["variations"])
            
            str_info["type"].setdefault(idx, []).append(concept["type"])
            str_info["summary"].setdefault(idx, []).append(concept["summary"])
            
        lbl_mat["indptr"].append(len(lbl_mat["indices"]))
        
    shape = (len(lbl_mat["indptr"]) - 1, len(lbl_mat["vocab"]))
    lbl_str = sp.csr_matrix((lbl_mat["data"], lbl_mat["indices"], lbl_mat["indptr"]), shape=shape, dtype=np.float32)
    
    return lbl_mat["vocab"], lbl_str, str_info
    

In [7]:
#| export
def get_matrix_from_dict(metadata:Dict, ids:List):
    vocab, data, indices, indptr = dict(), [], [], [0]
    for i in tqdm(ids):
        indices.extend([vocab.setdefault(o, len(vocab)) for o in metadata.get(i, [])])
        data.extend([1.0] * len(metadata.get(i, [])))
        indptr.append(len(indices))
    return sp.csr_matrix((data, indices, indptr), dtype=np.float32), vocab
    

#### Example

In [115]:
dataset = "msmarco"
data_dir = f"/Users/suchith720/Projects/data/beir/{dataset}/XC/"
lbl_ids, lbl_txt = load_raw_file(f"{data_dir}/raw_data/label.raw.txt")

In [136]:
str_vocab, lbl_str, str_info = get_metadata_matrix_from_label_dict(outputs, lbl_ids)

  0%|          | 0/8841823 [00:00<?, ?it/s]

In [137]:
str_txt = sorted(str_vocab, key=lambda x:str_vocab[x])
str_ids = list(range(len(str_txt)))

In [140]:
str_var, var_vocab = get_matrix_from_dict(str_info["variations"], str_ids)

  0%|          | 0/274 [00:00<?, ?it/s]

### Extract metadata

In [8]:
#| export
def extract_metadata(outputs:Dict, lbl_ids:List):
    o = get_metadata_matrix_from_label_dict(outputs, lbl_ids)

    str_vocab, lbl_str, str_info = get_metadata_matrix_from_label_dict(outputs, lbl_ids)

    str_txt = sorted(str_vocab, key=lambda x:str_vocab[x])
    str_ids = list(range(len(str_txt)))
    
    assert len(intent_ids) == lbl_str.shape[1], (
        f"Concept count mismatch: expected {lbl_str.shape[1]}, got {len(intent_ids)}"
    )

    str_type, type_vocab = get_matrix_from_dict(str_info["type"], str_ids)
    str_summary, summary_vocab = get_matrix_from_dict(str_info["summary"], str_ids)
    str_variations, variations_vocab = get_matrix_from_dict(str_info["variations"], str_ids)

    str_info = (str_type, type_vocab), (str_summary, summary_vocab), (str_variations, variations_vocab)
    
    return (str_ids, str_txt, lbl_str), str_info
    

In [9]:
#| export
def save_metadata(save_dir:str, concept_ids:List, concept_txt:List, lbl_concept:sp.csr_matrix):
    lbl_concept.sum_duplicates()
    lbl_concept.sort_indices()
    
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f"{save_dir}/raw_data/", exist_ok=True)
    
    sp.save_npz(f"{save_dir}/concept_lbl_X_Y.npz", lbl_concept)
    save_raw(f"{save_dir}/raw_data/label_concept.raw.csv", concept_ids, concept_txt)
    

In [6]:
#| export
def save_metadata_info(save_dir:str, str_info:Tuple):    
    (str_type, type_vocab), (str_summary, summary_vocab), (str_variations, variations_vocab) = str_info
    
    str_type.sum_duplicates()
    str_type.sort_indices()
    
    str_summary.sum_duplicates()
    str_summary.sort_indices()

    str_variations.sum_duplicates()
    str_variations.sort_indices()
    
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f"{save_dir}/raw_data/", exist_ok=True)

    def get_raw_info(vocab):
        txt = sorted(vocab, key=lambda x: vocab[x])
        ids = list(range(len(txt)))
        return ids, txt
    
    sp.save_npz(f"{save_dir}/variations_concept_X_Y.npz", str_variations)
    ids, txt = get_raw_info(variations_vocab)
    save_raw(f"{save_dir}/raw_data/concept_variations.raw.csv", ids, txt)
    
    sp.save_npz(f"{save_dir}/type_concept_X_Y.npz", str_type)
    ids, txt = get_raw_info(type_vocab)
    save_raw(f"{save_dir}/raw_data/concept_type.raw.csv", ids, txt)

    sp.save_npz(f"{save_dir}/summary_concept_X_Y.npz", str_summary)
    ids, txt = get_raw_info(summary_vocab)
    save_raw(f"{save_dir}/raw_data/concept_summary.raw.csv", ids, txt)
    

## `Driver`

In [ ]:
#| export
if __name__ == "__main__":
    output_dir = "/Users/suchith720/Downloads/outputs/"

    df = combine_df(data_dir)
    outputs = convert_string_to_object(df)

    for dataset in tqdm(BEIR_DATASETS):
        print(dataset)
        
        data_dir = f"/Users/suchith720/Projects/data/beir/{dataset}/XC/"
        lbl_ids, lbl_txt = load_raw_file(f"{data_dir}/raw_data/label.raw.csv")
        
        (str_ids, str_txt, lbl_str), str_info = extract_metadata(outputs, lbl_ids)
        
        save_dir = f"{data_dir}/document_concept_substring/"
        save_metadata(save_dir, str_ids, str_txt, lbl_str)
        save_metadata_info(save_dir, str_info)
            

### `intent-summary`

In [11]:
data_dir = "/Users/suchith720/Downloads/outputs/"

In [12]:
df = combine_df(data_dir)
outputs = convert_string_to_object(df)

  0%|          | 0/22 [00:00<?, ?it/s]

/var/folders/x0/r_2wlyls39s3_q5g99w33dn80000gn/T/ipykernel_19778/2824045793.py:5: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.concat([pd.read_table(f"{dirname}/{f}") for f in tqdm(files)], axis=0)


  0%|          | 0/2160403 [00:00<?, ?it/s]

# of empty generations: 79215.


In [ ]:
joblib.dump(outputs, f"{data_dir}/outputs.joblib")

In [ ]:
for dataset in tqdm(BEIR_DATASETS):
    print(dataset)
    
    data_dir = f"/Users/suchith720/Projects/data/beir/{dataset}/XC/"
    lbl_ids, lbl_txt = load_raw_file(f"{data_dir}/raw_data/label.raw.csv")
    
    (str_ids, str_txt, lbl_str), str_info = extract_metadata(outputs, lbl_ids)
    
    save_dir = f"{data_dir}/document_concept_substring/"
    save_metadata(save_dir, str_ids, str_txt, lbl_str)
    save_metadata_info(save_dir, str_info)